# Web Search & Content Extraction Tools — Demo

Quick demos of `google_search` and `fetch_page_as_markdown`.

In [1]:
import sys
sys.path.insert(0, '..')

from tools.web_search import GoogleSearchInput, GoogleSearchResult, google_search
from tools.web_search_brave import BraveSearchInput, BraveSearchResult, brave_search
from tools.web_search_ddg import DuckDuckGoSearchInput, DuckDuckGoSearchResult, duckduckgo_search
from tools.web_content import FetchPageInput, PageMarkdownResult, fetch_page_as_markdown
from IPython.display import Markdown, display

## 1. Fetch example.com — Simplest possible test

In [ ]:
args = FetchPageInput(url="https://example.com")
raw = fetch_page_as_markdown(args)
result = PageMarkdownResult.model_validate_json(raw)

print(f"Title: {result.title}")
print(f"Error: {result.error}")
print(f"Markdown length: {len(result.markdown)} chars")
print("---")
display(Markdown(result.markdown))

## 2. Google Search — "Pikachu pokemon"

In [ ]:
args = GoogleSearchInput(query="Pikachu pokemon", max_results=5)
raw = google_search(args)
result = GoogleSearchResult.model_validate_json(raw)

print(f"Query: {result.query}")
print(f"Error: {result.error}")
print(f"Results: {result.total_returned}\n")

for i, r in enumerate(result.results):
    print(f"[{i+1}] {r.title}")
    print(f"    {r.url}")
    if r.snippet:
        print(f"    {r.snippet[:120]}")
    print()

## 3. Google Search — Site-restricted to Bulbapedia

In [ ]:
args = GoogleSearchInput(
    query="Charizard",
    site_restrict="bulbapedia.bulbagarden.net",
    max_results=5,
)
raw = google_search(args)
result = GoogleSearchResult.model_validate_json(raw)

print(f"Query: {result.query}")
print(f"Results: {result.total_returned}\n")

for i, r in enumerate(result.results):
    print(f"[{i+1}] {r.title}")
    print(f"    {r.url}\n")

## 4. Fetch Bulbapedia — Pikachu page (stealth mode)

Bulbapedia is behind Cloudflare, so we use `use_stealth=True`.

In [ ]:
args = FetchPageInput(
    url="https://bulbapedia.bulbagarden.net/wiki/Pikachu_(Pok%C3%A9mon)",
    css_selector="#mw-content-text",
    use_stealth=True,
)
raw = fetch_page_as_markdown(args)
result = PageMarkdownResult.model_validate_json(raw)

print(f"Title: {result.title}")
print(f"Error: {result.error}")
print(f"Markdown length: {len(result.markdown):,} chars")
print("---")
# Show first 2000 chars as rendered markdown
display(Markdown(result.markdown[:20000] + "\n\n*... (truncated) ...*"))

## 5. Fetch Wikipedia — Clean content extraction

In [ ]:
args = FetchPageInput(
    url="https://en.wikipedia.org/wiki/Pok%C3%A9mon",
    css_selector="#bodyContent",
)
raw = fetch_page_as_markdown(args)
result = PageMarkdownResult.model_validate_json(raw)

print(f"Title: {result.title}")
print(f"Error: {result.error}")
print(f"Markdown length: {len(result.markdown):,} chars")
print("---")
display(Markdown(result.markdown[:3000] + "\n\n*... (truncated) ...*"))

## 6. Search → Fetch pipeline

Search for something, then fetch the first result and show its markdown.

In [ ]:
# Step 1: Search
search_args = GoogleSearchInput(
    query="Bulbasaur",
    site_restrict="bulbapedia.bulbagarden.net",
    max_results=1,
)
search_raw = google_search(search_args)
search_result = GoogleSearchResult.model_validate_json(search_raw)

if search_result.results:
    first = search_result.results[0]
    print(f"Top result: {first.title}")
    print(f"URL: {first.url}\n")

    # Step 2: Fetch that page
    fetch_args = FetchPageInput(
        url=first.url,
        css_selector="#mw-content-text",
        use_stealth=True,
    )
    fetch_raw = fetch_page_as_markdown(fetch_args)
    page = PageMarkdownResult.model_validate_json(fetch_raw)

    print(f"Page title: {page.title}")
    print(f"Content length: {len(page.markdown):,} chars")
    print("---")
    display(Markdown(page.markdown[:5000] + "\n\n*... (truncated) ...*"))
else:
    print("No results found.")

## 9. DuckDuckGo Search — "Bulbasaur site:bulbapedia.bulbagarden.net"

DuckDuckGo is used here as an alternative to Google Search, often providing better snippets for automated retrieval.

In [5]:
args = DuckDuckGoSearchInput(
    query="water pokemon",
    site_restrict="bulbapedia.bulbagarden.net",
    max_results=10
)
raw = duckduckgo_search(args)
result = DuckDuckGoSearchResult.model_validate_json(raw)

print(f"Query: {result.query}")
print(f"Results: {result.total_returned}\n")

for i, r in enumerate(result.results):
    print(f"[{i+1}] {r.title}")
    print(f"    {r.url}")
    if r.snippet:
        print(f"    {r.snippet[:2000]}...")
    print()

[2026-04-01 21:55:43] INFO: Fetched (200) <GET https://duckduckgo.com/?q=site%3Abulbapedia.bulbagarden.net+water+pokemon&t=h_&ia=web> (referer: https://www.google.com/)


Query: site:bulbapedia.bulbagarden.net water pokemon
Results: 10

[1] Water (type) - Bulbapedia, the community-driven Pokémon encyclopedia
    https://bulbapedia.bulbagarden.net/wiki/Water_(type)
    Mar 1, 2026 The Water type (Japanese: みずタイプ Water type) is one of the eighteen types. Water -type moves are super effective against Fire -, Ground -, and Rock-type Pokémon , while Water -type Pokémon are weak to Electric - and Grass-type moves....

[2] Category:Water-type Pokémon - Bulbapedia, the community-driven Pokémon ...
    https://bulbapedia.bulbagarden.net/wiki/Category:Water-type_Pok%C3%A9mon
    Pages in category " Water -type Pokémon " The following 159 pages are in this category, out of 159 total....

[3] Water 1 (Egg Group) - Bulbapedia
    https://bulbapedia.bulbagarden.net/wiki/Water_1_(Egg_group)
    Sep 17, 2025 Characteristics Water 1 group Pokémon tend to be mostly terrestrial Water -type Pokémon , with few fishlike Water types among them. Most are capable of both land-b

## 7. Vector Database Ingestion

Ingesting a webpage into the vector database using Chonkie for chunking and ChromaDB for storage.

In [ ]:
from tools.web_vector_db import ingest_web_page, IngestWebPageArgs

args = IngestWebPageArgs(
    url="https://bulbapedia.bulbagarden.net/wiki/Charmander_(Pok%C3%A9mon)",
    css_selector="#mw-content-text",
    use_stealth=True
)
result = ingest_web_page(args)
print(result)

## 8. Vector Database Querying

Retrieving semantically relevant chunks from the ingested web corpus.

In [ ]:
from tools.web_vector_db import query_web_content, QueryWebContentArgs

query_args = QueryWebContentArgs(
    query="What is the flame on Charmander's tail?",
    n_results=10
)
query_result = query_web_content(query_args)
display(Markdown(query_result))

## 10. Brave Search 

Brave Search is used for reliable, API-backed web search querying with optional `site_restrict` capabilities.

In [4]:
args = BraveSearchInput(
    query="Pikachu pokemon",
    site_restrict="bulbapedia.bulbagarden.net",
    max_results=10
)
raw = brave_search(args)
result = BraveSearchResult.model_validate_json(raw)

print(f"Query: {result.query}")
print(f"Results: {result.total_returned}\n")

for i, r in enumerate(result.results):
    print(f"[{i+1}] {r.title}")
    print(f"    {r.url}")
    if r.snippet:
        print(f"    {r.snippet[:1000]}")
    print()

Query: site:bulbapedia.bulbagarden.net Pikachu pokemon
Results: 10

[1] Pikachu (Pokémon) - Bulbapedia, the community-driven Pokémon encyclopedia
    https://bulbapedia.bulbagarden.net/wiki/Pikachu_(Pok%C3%A9mon)
    Pikachu is a short, chubby rodent Pokémon. It is covered in yellow fur with two horizontal brown stripes on its back. It has a small mouth, long, pointed ears with black tips, and brown eyes. Each cheek is a red circle that contains a pouch for electricity storage.

[2] Ash's Pikachu - Bulbapedia, the community-driven Pokémon encyclopedia
    https://bulbapedia.bulbagarden.net/wiki/Ash's_Pikachu
    <strong>Ash&#x27;s Pikachu</strong> (Japanese: サトシのピカチュウ Satoshi&#x27;s Pikachu) is the signature Pokémon of Pokémon the Series, and the first Pokémon that Ash obtained on his journey as a Pokémon Trainer, given to him by Professor Oak.

[3] Pikachu (Pokémon)/Generation III learnset - Bulbapedia, the community-driven Pokémon encyclopedia
    https://bulbapedia.bulbagarden.net/w

## 11. Brave Search — LLM Context API

Brave's [LLM Context API](https://api-dashboard.search.brave.com/documentation/services/llm-context) differs from regular web search by returning raw, pre-extracted content (like tables, markdown headers, and chunks of text) directly optimized for RAG rather than simple UI snippets. Let's see the deeper extraction format:

In [1]:
import sys
# Ensure the tools module is in python's path for the notebook
if '..' not in sys.path:
    sys.path.insert(0, '..')
    
from tools.web_search_brave_llm_context import BraveLLMContextInput, BraveLLMContextSearchResult, brave_llm_context_search

print("Running Brave LLM Context Search...\n")

# 1. Provide exact configuration parameters to the Pydantic schema
args = BraveLLMContextInput(
    query="Abra anime trainers",
    max_results=10,                           # Ask for top 2 URLs
    maximum_number_of_tokens=4096,           # Request up to ~2k tokens of data
    context_threshold_mode="balanced",       # Discards irrelevant data chunks
    site_restrict="bulbapedia.bulbagarden.net"  # Uncomment this to test site-scoping
)

# 2. Execute the wrapper tool
raw_json_response = brave_llm_context_search(args)

# 3. Validate and parse the response back into the Pydantic object
result = BraveLLMContextSearchResult.model_validate_json(raw_json_response)

print(f"Effective Query: {result.query}")
print(f"Total Results Extracted: {result.total_returned}\n")
print("=" * 60)

if result.error:
    print(f"Search Failed! Error: {result.error}")
else:
    for i, item in enumerate(result.results):
        print(f"\n[{i+1}] {item.title}")
        print(f"URL: {item.url}")
        
        # The 'snippet' field now safely holds the massive concatenated context blocks
        print(f"Extracted Chunk Size: {len(item.snippet):,} characters\n")
        
        print("--- Snippet Preview ---")
        # Print the first ~800 characters to show the deep formatting
        print(item.snippet)
        print("=" * 60)


Running Brave LLM Context Search...

Effective Query: site:bulbapedia.bulbagarden.net Abra anime trainers
Total Results Extracted: 6


[1] Abra (Pokémon) - Bulbapedia, the community-driven Pokémon ...
URL: https://bulbapedia.bulbagarden.net/wiki/Abra_(Pok%C3%A9mon)
Extracted Chunk Size: 4,372 characters

--- Snippet Preview ---
An Abra appeared in Classroom Training!, under the ownership of the Snowpoint Trainers' School. An Abra appeared in A Marathon Rivalry!, under the ownership of a competitor in the Pokéathlon held in Camellia Town. An Abra appeared in A Jolting Switcheroo!. An Abra appeared in Dreaming a Performer's Dream!. Three Abra appeared in Cloudy Fate, Bright Future!, with two under the ownership of different Psychics and the third under the ownership of a Trainer. Two Trainers' Abra appeared in Alola to New Adventure!. A Trainer's Abra appeared in A Shocking Grocery Run!. A Trainer's Abra appeared in Crystal-Clear Sleuthing!. Two Trainers' Abra appeared in One Journey End